In [0]:
from pyspark.sql import functions as F
import time, statistics

RUTA_CSV = "/Volumes/workspace/ef_precios_medicamentos/datos/sinteticos_1m.csv"
display(dbutils.fs.ls("/Volumes/workspace/ef_precios_medicamentos/datos/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/ef_precios_medicamentos/datos/datos_listos.csv,datos_listos.csv,20817424,1795285883000
dbfs:/Volumes/workspace/ef_precios_medicamentos/datos/sinteticos_100k.csv,sinteticos_100k.csv,23993101,1795314204000
dbfs:/Volumes/workspace/ef_precios_medicamentos/datos/sinteticos_1m.csv,sinteticos_1m.csv,240356190,1795314211000
dbfs:/Volumes/workspace/ef_precios_medicamentos/datos/sinteticos_500k.csv,sinteticos_500k.csv,120131575,1795314211000


In [0]:
bronze = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .option("quote", '"')
          .option("escape", '"')
          .csv(RUTA_CSV))

for c in bronze.columns:
    limpio = c.replace("\ufeff", "").strip()
    if limpio != c:
        bronze = bronze.withColumnRenamed(c, limpio)

bronze.write.mode("overwrite").saveAsTable("bronze_sinteticos")
print("Bronze filas:", spark.table("bronze_sinteticos").count())
bronze.printSchema()

Bronze filas: 1000000
root
 |-- codigo_producto: double (nullable = true)
 |-- nombre_comercial: string (nullable = true)
 |-- concentracion: string (nullable = true)
 |-- forma_farmaceutica: string (nullable = true)
 |-- presentacion: string (nullable = true)
 |-- registro_sanitario: string (nullable = true)
 |-- ifa: string (nullable = true)
 |-- titular: string (nullable = true)
 |-- fabricante: string (nullable = true)
 |-- pais: string (nullable = true)
 |-- clasificacion: string (nullable = true)
 |-- n_establecimientos: double (nullable = true)
 |-- precio_media: double (nullable = true)
 |-- precio_mediana: double (nullable = true)
 |-- precio_moda: double (nullable = true)
 |-- periodo: date (nullable = true)
 |-- mes: integer (nullable = true)
 |-- nombre_mes: string (nullable = true)
 |-- trimestre: integer (nullable = true)
 |-- anio: integer (nullable = true)
 |-- variacion_mensual_pct: double (nullable = true)



In [0]:
df = spark.table("bronze_sinteticos")

df.select("n_establecimientos", "precio_media", "precio_mediana",
          "precio_moda", "variacion_mensual_pct").describe().show()
df.groupBy("clasificacion").count().orderBy(F.desc("count")).show()
df.groupBy("mes", "nombre_mes").count().orderBy("mes").show()
df.groupBy("pais").count().orderBy(F.desc("count")).show(15, truncate=False)

+-------+------------------+------------------+------------------+------------------+---------------------+
|summary|n_establecimientos|      precio_media|    precio_mediana|       precio_moda|variacion_mensual_pct|
+-------+------------------+------------------+------------------+------------------+---------------------+
|  count|           1000000|           1000000|           1000000|           1000000|               816106|
|   mean|        760.080792|117.84499979469868|117.44838132619823|117.42761313779792|    1.045283149492834|
| stddev|1262.9925768027613|1244.8761027228006|1246.8900649470384| 1245.730182596255|    73.34897919593762|
|    min|               1.0|            0.0315|            0.0315|            0.0204|               -98.24|
|    max|            9921.0|        74629.9541|        74629.9541|        74629.9541|             12901.32|
+-------+------------------+------------------+------------------+------------------+---------------------+

+-------------+------+
|cla

In [0]:
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show(vertical=True)

-RECORD 0-----------------------
 codigo_producto       | 0      
 nombre_comercial      | 0      
 concentracion         | 32830  
 forma_farmaceutica    | 0      
 presentacion          | 0      
 registro_sanitario    | 0      
 ifa                   | 0      
 titular               | 0      
 fabricante            | 0      
 pais                  | 0      
 clasificacion         | 0      
 n_establecimientos    | 0      
 precio_media          | 0      
 precio_mediana        | 0      
 precio_moda           | 0      
 periodo               | 0      
 mes                   | 0      
 nombre_mes            | 0      
 trimestre             | 0      
 anio                  | 0      
 variacion_mensual_pct | 183894 



In [0]:
silver = (spark.table("bronze_sinteticos")
          .na.drop(subset=["codigo_producto", "precio_media"])
          .na.fill({"concentracion": "SIN DATO"})
          .filter((F.col("precio_media") > 0) & (F.col("n_establecimientos") > 0))
          .withColumnRenamed("n_establecimientos", "num_establecimientos")
          .withColumnRenamed("variacion_mensual_pct", "variacion_pct")
          .withColumnRenamed("anio", "anio_periodo")
          .select("codigo_producto", "nombre_comercial", "concentracion", "ifa",
                  "titular", "fabricante", "pais", "clasificacion",
                  "num_establecimientos", "precio_media", "precio_mediana",
                  "precio_moda", "periodo", "mes", "nombre_mes", "trimestre",
                  "anio_periodo", "variacion_pct"))

silver.write.mode("overwrite").saveAsTable("silver_sinteticos")
print("Silver filas:", spark.table("silver_sinteticos").count())
print("variacion_pct nulos:", spark.table("silver_sinteticos").filter(F.col("variacion_pct").isNull()).count())

Silver filas: 1000000
variacion_pct nulos: 183894


In [0]:
s = spark.table("silver_sinteticos")

gold_clasif = (s.groupBy("clasificacion")
               .agg(F.count("*").alias("registros"),
                    F.round(F.percentile_approx("precio_media", 0.5), 2).alias("precio_mediana"),
                    F.round(F.percentile_approx("variacion_pct", 0.5), 3).alias("variacion_mediana_pct")))
gold_clasif.write.mode("overwrite").saveAsTable("gold_agg_clasificacion")

gold_mes = (s.groupBy("mes", "nombre_mes")
            .agg(F.round(F.percentile_approx("precio_media", 0.5), 2).alias("precio_mediana"),
                 F.sum("num_establecimientos").cast("long").alias("total_establecimientos"))
            .orderBy("mes"))
gold_mes.write.mode("overwrite").saveAsTable("gold_agg_mes")

gold_pais = (s.groupBy("pais")
             .agg(F.count("*").alias("registros"),
                  F.round(F.percentile_approx("precio_media", 0.5), 2).alias("precio_mediana"))
             .orderBy(F.desc("registros")))
gold_pais.write.mode("overwrite").saveAsTable("gold_agg_pais")

for t in ["gold_agg_clasificacion", "gold_agg_mes", "gold_agg_pais"]:
    print(t)
    spark.table(t).show(10, truncate=False)

gold_agg_clasificacion
+-------------+---------+--------------+---------------------+
|clasificacion|registros|precio_mediana|variacion_mediana_pct|
+-------------+---------+--------------+---------------------+
|MARCA        |776558   |7.26          |0.0                  |
|GENERICO     |223442   |2.92          |0.0                  |
+-------------+---------+--------------+---------------------+

gold_agg_mes
+---+----------+--------------+----------------------+
|mes|nombre_mes|precio_mediana|total_establecimientos|
+---+----------+--------------+----------------------+
|1  |Enero     |5.53          |128708044             |
|2  |Febrero   |5.64          |128335070             |
|3  |Marzo     |5.62          |129430548             |
|4  |Abril     |5.49          |127825800             |
|5  |Mayo      |5.51          |121763933             |
|6  |Junio     |5.56          |124017397             |
+---+----------+--------------+----------------------+

gold_agg_pais
+---------+---------

In [0]:
spark.sql("SHOW TABLES").show(truncate=False)

+--------+----------------------+-----------+
|database|tableName             |isTemporary|
+--------+----------------------+-----------+
|default |bronze_sinteticos     |false      |
|default |gold_agg_clasificacion|false      |
|default |gold_agg_mes          |false      |
|default |gold_agg_pais         |false      |
|default |silver_sinteticos     |false      |
+--------+----------------------+-----------+



In [0]:
def medir(nombre, categoria, accion, reps=3):
    accion()
    tiempos = []
    for _ in range(reps):
        t0 = time.perf_counter()
        accion()
        tiempos.append(time.perf_counter() - t0)
    med = statistics.median(tiempos)
    print(f"{nombre:<28} {[round(t, 3) for t in tiempos]}  mediana = {med:.3f} s")
    return (nombre, categoria, round(tiempos[0], 3), round(tiempos[1], 3),
            round(tiempos[2], 3), round(med, 3), int(1_000_000 / med))

In [0]:
df_b = spark.table("bronze_sinteticos")
df_s = spark.table("silver_sinteticos")

operaciones = [
    ("1_lectura_csv", "I/O + parsing",
        lambda: spark.read.option("header", True).option("inferSchema", True)
                     .option("quote", '"').option("escape", '"').csv(RUTA_CSV).count()),
    ("2_filtro", "Narrow",
        lambda: df_b.filter((F.col("precio_media") > 50) & (F.col("clasificacion") == "MARCA")).count()),
    ("3_select_suma", "Narrow + suma global",
        lambda: df_b.select((F.col("precio_media") * 1.18).alias("p")).agg(F.sum("p")).collect()),
    ("4_describe", "Resumen estadistico",
        lambda: df_b.select("n_establecimientos", "precio_media", "precio_mediana", "precio_moda").describe().collect()),
    ("5_groupBy_clasif", "Wide (2 claves)",
        lambda: df_b.groupBy("clasificacion").count().collect()),
    ("6_groupBy_mes", "Wide (6 claves + sort)",
        lambda: df_s.groupBy("mes", "nombre_mes").agg(F.avg("precio_media"), F.sum("num_establecimientos")).orderBy("mes").collect()),
    ("7_groupBy_pais", "Wide (paises + sort)",
        lambda: df_s.groupBy("pais").agg(F.count("*").alias("n"), F.avg("precio_media")).orderBy(F.desc("n")).collect()),
    ("8_groupBy_producto", "Wide (alta cardinalidad)",
        lambda: df_s.groupBy("codigo_producto").agg(F.avg("precio_media")).count()),
    ("9_pipeline_limpieza", "Narrow encadenado",
        lambda: df_b.na.drop(subset=["codigo_producto", "precio_media"])
                    .na.fill({"concentracion": "SIN DATO"})
                    .filter((F.col("precio_media") > 0) & (F.col("n_establecimientos") > 0)).count()),
    ("10_escritura_delta", "Escritura Delta",
        lambda: df_s.write.mode("overwrite").saveAsTable("tmp_exp_escritura")),
]

resultados = []
for nombre, categoria, accion in operaciones:
    resultados.append(medir(nombre, categoria, accion))

1_lectura_csv                [2.056, 2.072, 2.216]  mediana = 2.072 s
2_filtro                     [0.607, 0.567, 0.614]  mediana = 0.607 s
3_select_suma                [0.473, 0.487, 0.489]  mediana = 0.487 s
4_describe                   [0.564, 0.632, 0.676]  mediana = 0.632 s
5_groupBy_clasif             [0.549, 0.492, 0.513]  mediana = 0.513 s
6_groupBy_mes                [0.447, 0.429, 0.491]  mediana = 0.447 s
7_groupBy_pais               [0.577, 0.433, 0.427]  mediana = 0.433 s
8_groupBy_producto           [0.378, 0.393, 0.373]  mediana = 0.378 s
9_pipeline_limpieza          [0.504, 0.563, 0.46]  mediana = 0.504 s
10_escritura_delta           [2.912, 2.836, 2.681]  mediana = 2.836 s


In [0]:
columnas = ["operacion", "categoria", "rep1_s", "rep2_s", "rep3_s", "mediana_s", "filas_por_s"]
df_res = spark.createDataFrame(resultados, columnas)
df_res.write.mode("overwrite").saveAsTable("exp_operaciones_resultados")
display(df_res)

operacion,categoria,rep1_s,rep2_s,rep3_s,mediana_s,filas_por_s
1_lectura_csv,I/O + parsing,2.056,2.072,2.216,2.072,482642
2_filtro,Narrow,0.607,0.567,0.614,0.607,1646778
3_select_suma,Narrow + suma global,0.473,0.487,0.489,0.487,2054179
4_describe,Resumen estadistico,0.564,0.632,0.676,0.632,1581964
5_groupBy_clasif,Wide (2 claves),0.549,0.492,0.513,0.513,1947488
6_groupBy_mes,Wide (6 claves + sort),0.447,0.429,0.491,0.447,2235778
7_groupBy_pais,Wide (paises + sort),0.577,0.433,0.427,0.433,2310454
8_groupBy_producto,Wide (alta cardinalidad),0.378,0.393,0.373,0.378,2646927
9_pipeline_limpieza,Narrow encadenado,0.504,0.563,0.46,0.504,1985091
10_escritura_delta,Escritura Delta,2.912,2.836,2.681,2.836,352636


In [0]:
display(df_s.groupBy("pais").agg(F.count("*").alias("n"), F.avg("precio_media")).orderBy(F.desc("n")))

pais,n,avg(precio_media)
PERU,479680,14.044417270055078
INDIA,164624,43.75890647961505
CHINA,66385,9.215768970399969
COLOMBIA,55707,13.99849059543659
ARGENTINA,29620,306.1920738352444
ALEMANIA,20248,1216.6277070426825
CHILE,19575,16.509157808429045
MEXICO,15997,31.991997512033393
ESPAÑA,15510,105.87870599613161
PARAGUAY,15386,49.00839975952192


In [0]:
spark.sql("DROP TABLE IF EXISTS tmp_exp_escritura")

DataFrame[]